# 05 · RQ5 rhythms: hour of day

**Question.** Does attention have a daily rhythm? When do lists churn fastest, when do new topics arrive, and when is activity highest? A creator who knows that a list turns over fastest at 9 a.m. Eastern knows when a new topic has the best chance to break in, and when a forecast has to look furthest ahead.

**Data.** The stored lists (`trend_items`), each list's top 10 at about :07 past every hour. Times are shown in US Eastern (`hour_local`); pass `clock="utc"` to `rhythms.by_hour` for UTC.

**Measures, per list and fetch.**
- *turnover*: the share of the top 10 that wasn't in the list about an hour earlier;
- *novelty*: items in the top 10 for the first time since the data starts;
- *flow*: how fast the top 10's metric grows per hour (posts, points, uses), relative to the list's median.

**What the lists are** (checked on 2026-09-30, see findings/rq5-rhythms.md):
- Google Trends' feed is the 10 *newest* US trends, newest first, not the 10 biggest. Its turnover is the rate at which new trends start, capped at 10 an hour, and its rank means age, not size.
- Every stored metric is a running total for its item, so activity is its growth between fetches.
- Items are matched across hours by url for Bluesky (it rewrites a topic's title as the story moves on) and Hacker News (titles get edited), and by title elsewhere.

**How much data it needs.** The hour-of-day effect is fitted with a separate baseline per day, so one busy news day isn't read as a daily rhythm. It needs at least 3 days per list; about a week gives each hour enough repeats to trust. Weekdays need about 3 weeks (3 of each weekday). Seasons need the Wikipedia pageview history (task 5.8).

Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from topnews import db, rhythms

engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = db.trend_items(engine, days=28)  # everything stored; YouTube left out
rhythms.coverage(items)

## Which lists count

TikTok refreshes daily and Instagram every 6 hours; in between, the same list repeats every hour, so their hour-to-hour changes aren't signal. They're left out here.

In [ ]:
fast = items[~items.source_id.isin(rhythms.SLOW_SOURCES)]
lists = rhythms.list_runs(fast)
lists.groupby("list_id").agg(fetches=("run_id", "size"), first=("fetched_at", "min"), last=("fetched_at", "max"))

In [ ]:
def by_hour_chart(df, value, title, ylabel):
    """One panel per list: each fetch as a dot, the hourly mean as a line (US Eastern)."""
    names = sorted(df.list_id.unique())
    fig, axes = plt.subplots(1, len(names), figsize=(3 * len(names), 2.6), sharey=True, squeeze=False)
    for ax, name in zip(axes[0], names):
        part = df[df.list_id == name]
        ax.scatter(part.hour_local, part[value], s=10, alpha=0.5)
        means = part.groupby("hour_local")[value].mean()
        ax.plot(means.index, means.values, lw=1.5)
        ax.set_title(name, fontsize=9)
        ax.set_xlim(-0.5, 23.5)
        ax.set_xticks([0, 6, 12, 18])
        ax.set_xlabel("hour, US Eastern", fontsize=8)
    axes[0][0].set_ylabel(ylabel)
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    plt.show()

## Turnover: how fast each list churns

Each fetch is compared with the fetch about an hour earlier (an extra manual run doesn't shorten the window, and a missed run doesn't stretch it). For Google, 1.0 means 10 or more new trends that hour: the feed can't show more.

In [ ]:
turn = rhythms.with_clock(rhythms.turnover(lists))
turn.groupby("list_id").agg(
    comparisons=("share_new", "size"),
    mean_share_new=("share_new", "mean"),
    median_new=("new", "median"),
    max_new=("new", "max"),
    hours_with_no_change=("new", lambda s: int((s == 0).sum())),
).round(2)

In [ ]:
by_hour_chart(turn, "share_new", "Share of the top 10 that is new since an hour earlier", "share new")
rhythms.by_hour(turn, "share_new").pivot(index="hour", columns="list_id", values="mean").round(2)

## Novelty: when new topics arrive

Turnover counts re-entries too (a topic that drops to #11 and comes back). Novelty counts only a topic's first appearance in the top 10. Early hours run high, because a topic that trended just before the data starts looks new when it returns; the bias fades as the data grows.

In [ ]:
nov = rhythms.with_clock(rhythms.novelty(fast))
by_hour_chart(nov, "first_time", "Topics in the top 10 for the first time", "new topics per fetch")
rhythms.by_hour(nov, "first_time").pivot(index="hour", columns="list_id", values="mean").round(1)

## Flow: how much activity the top 10 carries

The median hourly growth of the metric across top-10 items that were also in the list an hour earlier:
- Bluesky: new posts about the topic per hour;
- Hacker News: new points per hour on front-page stories;
- Mastodon: new uses of the tag per hour (the count restarts at midnight UTC; that hour drops out).

Google's searches come in coarse buckets (200+, 500+, 1,000+ …), too coarse for growth, and Twitch's list has no viewer count, so neither is here. The raw totals would mostly measure how old the list's items are: in the morning Bluesky's top 10 still held the previous day's big stories.

In [ ]:
flo = rhythms.with_clock(rhythms.flow(fast))
flo.groupby("list_id").agg(fetches=("rate", "size"), median_items_matched=("matched", "median"), median_rate_per_hour=("rate", "median")).round(1)

In [ ]:
by_hour_chart(flo, "relative", "Hourly growth of the top 10's metric, relative to the list's median", "× list median")
rhythms.by_hour(flo, "relative").pivot(index="hour", columns="list_id", values="mean").round(2)

## Is there an hour-of-day effect?

`value ~ C(hour) + C(day)` per list, by ordinary least squares. `f_pvalue` tests whether all hour effects are zero; `peak_minus_low` is the gap between the highest and lowest hour, in the measure's own units. With 5 lists and 3 measures, expect about one p-value below 0.05 by chance alone, so a rhythm counts only when it's below 0.01 and holds in the following week too.

In [ ]:
for label, df, value in [("turnover", turn, "share_new"), ("novelty", nov, "first_time"), ("flow", flo, "relative")]:
    result = rhythms.hour_effect(df, value)
    days = df.groupby("list_id").date_local.nunique()
    if result is None:
        print(f"{label}: not yet. Needs {rhythms.MIN_DAYS} days per list; the most any list has is {days.max()}.")
    else:
        print(label)
        display(result.round(3))

## Weekdays (later)

A weekday effect needs about 3 of each weekday, so about 3 weeks of data, fitted with a separate baseline per week. Until then, this only counts the days on hand.

In [ ]:
days = rhythms.with_clock(lists).drop_duplicates(["list_id", "date_local"])
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
days.groupby("weekday_local").date_local.nunique().reindex(order, fill_value=0).rename("days on hand")